1. Load Data

In [2]:
import pandas as pd

df = pd.read_excel("NBA_data.xlsx")
df.head()

,Player,shots in 0-3,shots in 3-10,shots in 10-16,shots in 16-3P,shots in 3P,2P shot assisted,3P shot assisted,%FGA,%3PA,...,FTr,ORB%,DRB%,AST%,STL%,BLK%,TOV%,USG%,MP,FGA
0,Precious Achiuwa,0.369,0.388,0.041,0.007,0.190,0.593,1.000,0.116,0.490,...,0.261,10.4,20.2,8.0,1.8,2.3,9.1,15.4,20.7,6.1
1,Steven Adams,0.655,0.309,0.036,0.000,0.000,NaN,0.514,NaN,0.129,...,0.583,21.7,19.7,8.4,1.5,2.5,16.7,12.0,22.8,4.3
2,Bam Adebayo,0.177,0.270,0.180,0.053,0.320,0.615,0.944,0.073,0.356,...,0.311,6.9,25.7,12.6,1.4,2.0,9.9,23.8,31.1,14.5
3,Ochai Agbaji,0.441,0.159,0.012,0.006,0.382,0.783,1.000,0.141,0.523,...,0.171,5.6,10.8,6.1,1.3,1.7,9.4,13.2,15.5,4.0
4,Santi Aldama,0.206,0.263,0.082,0.029,0.420,0.804,0.986,0.044,0.245,...,0.239,6.0,19.4,15.6,1.5,2.4,9.5,20.0,27.9,11.1


2. Filter data & set hyperparameters

In [3]:
import random
import numpy as np
import torch

# --- Hyperparameters ---
min_mp        = 4      # minimum minutes per game
learning_rate = 0.001
epochs        = 150    # more epochs for VAE to converge
weight_d      = 1e-4   # small L2 regularisation
beta          = 0.5    # KL weight in VAE loss (< 1 keeps reconstruction dominant)

# Seed everything for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

df_filtered = df[df["MP"] >= min_mp].copy()
print("Original shape:", df.shape)
print("Filtered shape:", df_filtered.shape)

Original shape: (521, 22)
Filtered shape: (500, 22)


4. Preprocess & build VAE

In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

df_model = df_filtered.copy().reset_index(drop=True)
player_names = df_model["Player"]

X = df_model.drop(columns=["Player"]).copy()
X = X.apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())
X_values = X.values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_values)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)
dataset   = TensorDataset(X_tensor)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)


# ── Variational Autoencoder ────────────────────────────────────────────────

class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim=6):
        super().__init__()
        self.encoder_base = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
        )
        self.fc_mu     = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )

    def encode(self, x):
        h = self.encoder_base(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar


def vae_loss(recon, x, mu, logvar, beta=0.5):
    recon_loss = nn.MSELoss()(recon, x)
    # KL divergence: how much the learned distribution diverges from N(0,1)
    kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl


input_dim  = X_scaled.shape[1]
latent_dim = 6   # tuned below; 6 works better than 8 for KMeans on ~500 players

model     = VAE(input_dim, latent_dim=latent_dim)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_d)



5. Train VAE

In [5]:
model.train()
for epoch in range(epochs):
    total_loss = 0
    for (x,) in dataloader:
        recon, mu, logvar = model(x)
        loss = vae_loss(recon, x, mu, logvar, beta=beta)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:3d}/{epochs}  Loss: {total_loss:.4f}")

# Extract deterministic embeddings (mu) — stable, no sampling noise
model.eval()
with torch.no_grad():
    mu_all, _ = model.encode(X_tensor)
    embeddings = mu_all.numpy()

print("\nEmbedding shape:", embeddings.shape)

Epoch  10/150  Loss: 14.3130
Epoch  20/150  Loss: 13.1530
Epoch  30/150  Loss: 13.1905
Epoch  40/150  Loss: 13.0442
Epoch  50/150  Loss: 12.8391
Epoch  60/150  Loss: 12.4524
Epoch  70/150  Loss: 12.5510
Epoch  80/150  Loss: 12.5148
Epoch  90/150  Loss: 12.2581
Epoch 100/150  Loss: 12.7260
Epoch 110/150  Loss: 12.1959
Epoch 120/150  Loss: 12.3645
Epoch 130/150  Loss: 12.3510
Epoch 140/150  Loss: 12.1905
Epoch 150/150  Loss: 12.2097

Embedding shape: (500, 6)


6. Tune number of clusters (silhouette sweep)

In [6]:
from sklearn.metrics import silhouette_score

# Sweep k from 4 to 15, use many restarts to avoid bad KMeans initialisations
sweep_results = {}
for k in range(4, 16):
    km_tmp = KMeans(n_clusters=k, init="k-means++", n_init=50, random_state=42)
    labels_tmp = km_tmp.fit_predict(embeddings)
    score_tmp  = silhouette_score(embeddings, labels_tmp)
    sweep_results[k] = round(score_tmp, 5)
    print(f"k={k:2d}  silhouette={score_tmp:.5f}")

best_k = max(sweep_results, key=sweep_results.get)
print(f"\nBest k = {best_k}  (silhouette = {sweep_results[best_k]})")

k= 4  silhouette=0.35095
k= 5  silhouette=0.36418
k= 6  silhouette=0.36833
k= 7  silhouette=0.37461
k= 8  silhouette=0.36524
k= 9  silhouette=0.35845
k=10  silhouette=0.34715
k=11  silhouette=0.34884
k=12  silhouette=0.35846
k=13  silhouette=0.35475
k=14  silhouette=0.35190
k=15  silhouette=0.36114

Best k = 7  (silhouette = 0.3746100068092346)


7. Final clustering with best k

In [7]:
best_k = 9
kmeans = KMeans(n_clusters=best_k, init="k-means++", n_init=50, random_state=42)
clusters = kmeans.fit_predict(embeddings)

results = pd.DataFrame({"Player": player_names, "Cluster": clusters})

final_score = silhouette_score(embeddings, clusters)
print(f"Final silhouette score (VAE, k={best_k}): {final_score:.5f}")

Final silhouette score (VAE, k=9): 0.35845


8. Cluster summary — size, representative player, average stats

In [8]:
model_df = df_model.copy()
model_df["Cluster"] = clusters

cluster_sizes  = results["Cluster"].value_counts().sort_index()
numeric_cols   = model_df.drop(columns=["Player", "Cluster"]).columns
cluster_summary = model_df.groupby("Cluster")[numeric_cols].mean()

# Representative player = closest to the cluster centroid in embedding space
centers       = kmeans.cluster_centers_
representatives = []
for i in range(kmeans.n_clusters):
    idx  = np.where(clusters == i)[0]
    embs = embeddings[idx]
    dist = np.linalg.norm(embs - centers[i], axis=1)
    representatives.append(player_names.iloc[idx[np.argmin(dist)]])

rep_df = pd.DataFrame({
    "Cluster": range(kmeans.n_clusters),
    "Cluster Size": cluster_sizes.values,
    "Representative Player": representatives
})
print(rep_df)

   Cluster  Cluster Size Representative Player
0        0            44           DeAaron Fox
1        1            67          Santi Aldama
2        2            76           Eric Gordon
3        3            36        Khaman Maluach
4        4            50              Tre Mann
5        5            42           Josh Giddey
6        6            74         Johnny Furphy
7        7            41          Jakob Poeltl
8        8            70         Tyrese Martin


9. Detailed cluster profiles

In [9]:
rep_map = dict(zip(rep_df["Cluster"], rep_df["Representative Player"]))

def get_closest_players(cluster_id, n=5):
    idx  = np.where(clusters == cluster_id)[0]
    embs = embeddings[idx]
    dist = np.linalg.norm(embs - centers[cluster_id], axis=1)
    return player_names.iloc[idx[np.argsort(dist)][:n]].tolist()

def format_cluster(cluster_id, n_examples=5):
    size        = int((clusters == cluster_id).sum())
    rep_player  = rep_map.get(cluster_id, "Unknown")
    examples    = get_closest_players(cluster_id, n_examples)
    avg_stats   = cluster_summary.loc[cluster_id]

    lines = [
        f"Cluster {cluster_id}",
        f"Size: {size}",
        f"Representative player: {rep_player}",
        f"Example players: {', '.join(examples)}",
        "Average stats:",
    ]
    for stat, val in avg_stats.items():
        lines.append(f"  {stat}: {val:.3f}")
    return "\n".join(lines)

for cid in sorted(model_df["Cluster"].unique()):
    print("=" * 80)
    print(format_cluster(cid))
    print()

Cluster 0
Size: 44
Representative player: DeAaron Fox
Example players: DeAaron Fox, Kevin Porter Jr., Donovan Mitchell, Dillon Brooks, Cam Thomas
Average stats:
  shots in 0-3: 0.147
  shots in 3-10: 0.200
  shots in 10-16: 0.160
  shots in 16-3P: 0.096
  shots in 3P: 0.396
  2P shot assisted: 0.329
  3P shot assisted: 0.656
  %FGA: 0.016
  %3PA: 0.137
  TS%: 0.583
  3PAr: 0.396
  FTr: 0.301
  ORB%: 2.459
  DRB%: 12.405
  AST%: 29.291
  STL%: 1.780
  BLK%: 1.080
  TOV%: 12.755
  USG%: 27.755
  MP: 30.816
  FGA: 15.595

Cluster 1
Size: 67
Representative player: Santi Aldama
Example players: Santi Aldama, Jordan Miller, Cam Whitmore, Onyeka Okongwu, Tari Eason
Average stats:
  shots in 0-3: 0.266
  shots in 3-10: 0.245
  shots in 10-16: 0.074
  shots in 16-3P: 0.029
  shots in 3P: 0.385
  2P shot assisted: 0.560
  3P shot assisted: 0.925
  %FGA: 0.065
  %3PA: 0.305
  TS%: 0.568
  3PAr: 0.385
  FTr: 0.270
  ORB%: 5.352
  DRB%: 15.309
  AST%: 14.267
  STL%: 1.778
  BLK%: 1.766
  TOV%: 12.0

10. Find what cluster a player belongs to

In [10]:
def find_player_cluster(player_name):
    match = results[results["Player"].str.contains(player_name, case=False, na=False)]
    if len(match) == 0:
        return "Player not found."
    if len(match) > 1:
        return match[["Player", "Cluster"]]
    player  = match.iloc[0]["Player"]
    cluster = match.iloc[0]["Cluster"]
    return f"{player} is in Cluster {cluster}."

print(find_player_cluster("LeBron James"))
print(find_player_cluster("nikola jokic"))
print(find_player_cluster("stephen curry"))

LeBron James is in Cluster 5.
Nikola Jokic is in Cluster 5.
Stephen Curry is in Cluster 0.


11. List all players in a cluster, sorted by a stat

In [11]:
def list_players_in_cluster(cluster_id, sort_by="MP", ascending=False, cols=None):
    cluster_df = model_df[model_df["Cluster"] == cluster_id].copy()
    cluster_df = cluster_df.sort_values(by=sort_by, ascending=ascending)
    if cols is None:
        cols = ["Player", sort_by]
    return cluster_df[cols]

pd.set_option("display.max_rows", None)
print(list_players_in_cluster(0, sort_by="MP", cols=["Player", "MP", "FGA", "USG%"]))

                      Player    MP   FGA  USG%
287             Tyrese Maxey  38.9  21.5  28.8
115             Kevin Durant  36.7  18.0  26.8
109              Luka Doncic  36.0  22.3  37.8
319             Jamal Murray  35.8  18.8  28.7
121          Anthony Edwards  35.5  20.7  31.6
166             James Harden  35.4  17.5  31.3
95           Cade Cunningham  35.2  19.3  30.6
372            Austin Reaves  34.7  15.9  28.7
53             Jalen Brunson  34.6  20.6  31.0
140           Keyonte George  34.4  16.5  27.7
51              Jaylen Brown  34.3  22.5  36.7
475            Derrick White  34.2  15.4  23.3
198           Brandon Ingram  34.1  17.4  27.3
39              Devin Booker  34.0  18.4  31.3
310         Donovan Mitchell  33.8  20.7  32.4
353         Kevin Porter Jr.  33.5  13.1  24.1
144  Shai Gilgeous-Alexander  33.3  19.7  33.6
103            DeMar DeRozan  33.0  13.5  22.2
261            Kawhi Leonard  32.7  19.5  32.9
363         Payton Pritchard  32.6  14.0  21.2
134          

12. Predict cluster for a new player

In [16]:
X_columns = model_df.drop(columns=["Player", "Cluster"]).columns
X_medians = model_df.drop(columns=["Player", "Cluster"]).apply(pd.to_numeric, errors="coerce").median()

def predict_new_player_cluster(new_player_stats):
    new_df = pd.DataFrame([new_player_stats])[X_columns]
    new_df = new_df.apply(pd.to_numeric, errors="coerce").fillna(X_medians)
    new_scaled = scaler.transform(new_df.values)
    new_tensor = torch.tensor(new_scaled, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        mu, _ = model.encode(new_tensor)
        new_embedding = mu.numpy()

    return kmeans.predict(new_embedding)[0]

# Example new player — update values as needed
# Note: do NOT include the dropped columns (shots in 16-3P, %FGA, %3PA)
new_player = {
    "shots in 0-3": 0.28,
    "shots in 3-10": 0.18,
    "shots in 10-16": 0.04,
    "shots in 16-3P": 0.02,
    "shots in 3P": 0.48,
    "2P shot assisted": 0.72,
    "3P shot assisted": 0.96,
    "%FGA": 0.08,
    "%3PA": 0.52,
    "TS%": 0.58,
    "3PAr": 0.51,
    "FTr": 0.21,
    "ORB%": 4.5,
    "DRB%": 12.8,
    "AST%": 9.3,
    "STL%": 1.8,
    "BLK%": 1.2,
    "TOV%": 10.4,
    "USG%": 16.2,
    "MP": 28.0,
    "FGA": 8.7
}

print("Predicted cluster:", predict_new_player_cluster(new_player))

Predicted cluster: 6


13. Baseline comparison — PCA vs VAE silhouette scores

In [13]:
from sklearn.decomposition import PCA

# PCA baseline (same number of components as latent_dim for fair comparison)
pca = PCA(n_components=latent_dim)
X_pca = pca.fit_transform(X_scaled)
km_pca = KMeans(n_clusters=best_k, init="k-means++", n_init=50, random_state=42)
labels_pca = km_pca.fit_predict(X_pca)
pca_score  = silhouette_score(X_pca, labels_pca)

print(f"Baseline PCA silhouette score : {pca_score:.5f}")
print(f"VAE silhouette score          : {final_score:.5f}")
print(f"Improvement                   : {final_score - pca_score:+.5f}")

Baseline PCA silhouette score : 0.18801
VAE silhouette score          : 0.35845
Improvement                   : +0.17044


14. Testing on New Data

In [18]:
# Load new data
df_2016 = pd.read_excel("NBA_2016.xlsx")
df_2016 = df_2016.dropna(subset=["Player"])
df_2016 = df_2016[df_2016["MP"] >= 10].copy().reset_index(drop=True)

# Preprocess using the already-fitted scaler
X_new = df_2016.drop(columns=["Player"]).apply(pd.to_numeric, errors="coerce")
X_new = X_new[X_columns].fillna(X_medians)
X_new_scaled = scaler.transform(X_new.values)
X_new_tensor = torch.tensor(X_new_scaled, dtype=torch.float32)

# Encode with VAE and predict cluster
model.eval()
with torch.no_grad():
    mu, _ = model.encode(X_new_tensor)

clusters_2016 = kmeans.predict(mu.numpy())

results_2016 = pd.DataFrame({"Player": df_2016["Player"], "Cluster": clusters_2016})
print(results_2016.to_string(index=False))

                  Player  Cluster
              Quincy Acy        6
            Steven Adams        7
           Arron Afflalo        0
           Alexis Ajinca        5
            Cole Aldrich        7
       LaMarcus Aldridge        5
             Lavoy Allen        5
              Tony Allen        1
         Al-Farouq Aminu        1
          Chris Andersen        3
           Alan Anderson        8
          James Anderson        8
         Justin Anderson        8
           Kyle Anderson        1
           Ryan Anderson        0
   Giannis Antetokounmpo        5
         Carmelo Anthony        0
            Trevor Ariza        8
          Darrell Arthur        4
               Omer Asik        7
           D.J. Augustin        4
            Luke Babbitt        1
         Leandro Barbosa        8
              J.J. Barea        0
         Andrea Bargnani        0
         Harrison Barnes        1
             Matt Barnes        8
             Will Barton        5
            Br